# Adam can converge to the **wrong** point

### The Reddi-Kale-Kumar (2018) counterexample, as a *fixed* convex problem

> S. J. Reddi, S. Kale, S. Kumar. *On the Convergence of Adam and Beyond.* ICLR 2018 (best paper).

Adam's original convergence proof was **flawed**. Reddi et al. exhibited a simple convex problem
on which Adam provably converges to the *worst* point of the domain, and proposed **AMSGrad** as a
fix. The original statement is in the online / regret setting, but it is really just **SGD on a
fixed convex finite sum** (an ERM problem) - which is all we need here. This notebook reproduces the
failure in one dimension.

*MasterMath - Continuous Optimization, Lecture 14 (Neural Networks & Automatic Differentiation).*
*Runs as-is on Google Colab (only `numpy` and `matplotlib`, both preinstalled).*

## The problem - a fixed convex finite sum

One dimension, with $x$ constrained to $[-1, 1]$. Take **three training examples** with component
losses (for a constant $C>2$)

$$ f_1(x) = C\,x, \qquad f_2(x) = -x, \qquad f_3(x) = -x, $$

and minimise their average - the (fixed, convex, linear) **empirical risk**

$$ f(x) = \tfrac13\bigl(f_1(x)+f_2(x)+f_3(x)\bigr) = \frac{C-2}{3}\,x,
   \qquad\Longrightarrow\qquad x^\star = -1. $$

We minimise $f$ by **SGD**: each step samples one example and uses its gradient, cycling
$i = 1,2,3,1,\dots$. The per-step gradients are therefore periodic,

$$ g_t = \begin{cases} +C & t \equiv 1 \pmod 3 \quad(\text{large, } \textbf{rare})\\
                       -1 & \text{otherwise} \quad(\text{small, } \textbf{frequent}). \end{cases} $$

The rare large gradient (example 1) is the *informative* one: it points toward the optimum.

- **SGD** uses the raw gradient: the rare $+C$ step outweighs the two $-1$ steps $\Rightarrow x\to-1$. (correct)
- **Adam** divides by $\sqrt{v}$, normalising the rare large gradient down to $\approx 1$, so it can no
  longer outvote the two frequent small gradients $\Rightarrow x\to +1$ - the **maximiser** of a convex
  function on the box. (wrong corner)
- **AMSGrad** uses $\hat v_t=\max$-so-far, keeping the denominator large after the big gradient, which
  restores convergence to $-1$. (correct)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def stochastic_gradient(t, C):
    # SGD over the 3-term finite sum f = (f1+f2+f3)/3, cycling i = 1,2,3,1,...
    # example 1 has gradient C; examples 2 and 3 have gradient -1.
    return C if (t % 3 == 1) else -1.0

In [ ]:
def optimize(method, T, C, lr=0.02, beta1=0.0, beta2=0.7, eps=1e-8):
    # method in {'sgd','adam','amsgrad'}; returns the trajectory x_t over T steps
    x = m = v = vhat_max = 0.0
    xs = np.empty(T)
    for t in range(1, T + 1):
        g = stochastic_gradient(t, C)
        m = beta1 * m + (1 - beta1) * g           # 1st moment (EMA of gradient)
        v = beta2 * v + (1 - beta2) * g * g        # 2nd moment (EMA of squared gradient)
        m_hat = m / (1 - beta1**t) if beta1 > 0 else m
        v_hat = v / (1 - beta2**t)
        if method == 'sgd':
            x -= lr * g
        elif method == 'adam':
            x -= lr * m_hat / (np.sqrt(v_hat) + eps)
        elif method == 'amsgrad':
            vhat_max = max(vhat_max, v_hat)
            x -= lr * m_hat / (np.sqrt(vhat_max) + eps)
        x = min(1.0, max(-1.0, x))                # project onto [-1, 1]
        xs[t - 1] = x
    return xs

## Run the three methods

We pick $C=2.2$ (so the optimum is genuinely $x^\star=-1$), with $\beta_1=0$ and $\beta_2=0.7$
(see the caveats at the end for why).

In [ ]:
C, T = 2.2, 3000
plt.figure(figsize=(7, 4.2))
for method, color in [('adam', 'C3'), ('sgd', 'C0'), ('amsgrad', 'C2')]:
    plt.plot(optimize(method, T, C), color=color, lw=1.8, label=method)
plt.axhline(-1, ls='--', color='gray', lw=1)
plt.text(T*0.45, -0.93, r'optimum $x^\star=-1$', color='gray')
plt.xlabel('iteration'); plt.ylabel(r'$x_t$'); plt.ylim(-1.1, 1.1)
plt.title(rf'Fixed convex ERM ($C={C}$, $\beta_1=0$, $\beta_2=0.7$)')
plt.legend(); plt.show()

for m in ('adam', 'sgd', 'amsgrad'):
    print(f'final x [{m:>7}] = {optimize(m, 8000, C)[-1]:+.3f}')

**Adam climbs to $+1$ - the wrong corner - while SGD and AMSGrad descend to the optimum $-1$.**

## A whole *band* of $C$, not a single fluke

Sweep $C$ and record where Adam ends up: there is a clear interval just above $C=2$ where Adam is
attracted to $+1$.

In [ ]:
Cs = np.linspace(2.0, 3.0, 60)
finals = [optimize('adam', 8000, c)[-1] for c in Cs]
plt.figure(figsize=(7, 4.2))
plt.axhline(0, color='gray', lw=0.6)
plt.plot(Cs, finals, 'o-', ms=3, color='C3')
plt.xlabel('C'); plt.ylabel(r"Adam's final $x$")
plt.title(r'Adam $\to +1$ over a band of $C$'); plt.show()

## Caveats (so you trust this for the right reasons)

- **Why $\beta_1=0$?** It isolates the adaptive denominator $1/\sqrt{v}$ - the actual source of the
  pathology (this matches the paper's deterministic Theorem 1). With momentum ($\beta_1=0.9$) the moving
  average can *mask* the effect on this particular cyclic sampling order. The paper's **randomised**
  sampling construction breaks Adam for any admissible $(\beta_1,\beta_2)$ - try replacing the cyclic
  gradient with `g = C if rng.random() < p else -1.0`.
- **Why $\beta_2=0.7$?** It lets $v$ decay fast enough to expose the failure within a few thousand steps.
  The same pathology exists at the default $\beta_2=0.999$, but only for larger $C$ and far longer horizons.
- **Takeaway:** vanilla Adam has *no* general convergence guarantee. With $\beta_2\to 1$ and a suitable
  step size it provably reaches a **stationary point** of a smooth nonconvex objective at rate
  $O(1/\sqrt{T})$ - the *same* rate as SGD, and only to a stationary point, never a guaranteed minimum.
  AMSGrad restores the convergence guarantee.